In [42]:
from Z import *
from Poly import *
import numpy as np

In [ ]:
''' Some number theoretic helper functions'''
def factor(n : int) -> list[list[int]]:
    assert n >= 1, 'factoring is defined for positive integers'
    result = []
    num = n
    with open('small_primes.txt') as file:
        for prime in file:
            p = int(prime[:-1])
            if num % p == 0:
                div = [p, 1]
                num = num // p
                while num % p == 0:
                    num //= p
                    div[1] += 1
                result.append(div)
            if num == 1: 
                return result
        raise ValueError('prime factors are too big to factor by trial division')
    
def mobius(n : int) -> int:
    assert n >= 1, 'mobius function is defined for positive integers'
    factors = factor(n)
    if any([exp > 1 for prime, exp in factors]):
        return 0
    else:
        return (-1) ** (len(factors) % 2)

def phi(n : int) -> int:
    assert n >= 1, 'totient function is defined for positive integers'
    factors = factor(n)
    result = 1
    for prime, exp in factors:
        result *= (prime - 1) * prime ** (exp - 1)
    return result

def rad(n : int) -> int:
    assert n >= 1, 'radical function is defined for positive integers'
    factors = factor(n)
    result = 1
    for prime, exp in factors:
        result *= prime
    return result

def divs(n : int) -> list[int]:
    assert n >= 1, 'divisors are defined for positive integers'
    if n == 1: return [1]
    factors = factor(n)
    p, e = factors[0]
    divisors = [p ** k for k in range(e + 1)]
    for prime, exp in factors[1:]:
        more_divs = [prime ** k * div for div in divisors 
                     for k in range(1, exp + 1)]
        divisors += more_divs
    return sorted(divisors)

In [ ]:
''' Some specific polynomial operations as arrays'''
def compose(poly : np.ndarray, exp : int) -> np.ndarray:
    ''' poly(x) -> poly(x ^ exp)'''
    d = len(poly) - 1
    result = np.zeros(exp * d + 1, dtype=object) # because 64 bits too small ;)
    result[::exp] = poly
    return result

def unity_mult_mod(poly: np.ndarray, n : int, d : int) -> np.ndarray:
    ''' poly(x) -> poly(x)(1 - x^n) modulo x^d'''
    # make the right length first
    result = poly[:d] if len(poly) >= d else np.pad(poly, (0, d - len(poly)))
    for i in range(d - 1, n - 1, -1):
        result[i] -= result[i - n]
    return result

def unity_div_mod(poly : np.ndarray, n : int, d : int) -> np.ndarray:
    ''' poly(x) -> poly(x)/(1 - x^n) modulo x^d'''
    # make the right length first
    result = poly[:d] if len(poly) >= d else np.pad(poly, (0, d - len(poly)))
    for i in range(n, d):
        result[i] += result[i - n]
    return result

In [ ]:
def cyclo(n : int) -> np.ndarray:
    